# SSUSA–IUCN Spatial Join Pipeline

This notebook creates spatial footprints around SSUSA camera locations and intersects them
with cleaned IUCN range polygons.

The workflow produces:

- camera-level 1 km footprints
- a lightweight camera-to-species lookup table
- array-level union footprints created by dissolving camera buffers within arrays
- a lightweight array-to-species lookup table

#### Note
To avoid very large geospatial outputs, the spatial join results are saved as lookup tables
without repeated IUCN geometries.


In [91]:
import os
import math
import pandas as pd
import geopandas as gpd

In [92]:
# Paths and parameters
# ----------------------------------------------------------
CLEANED_PATH = "cleaned"
OUTPUT_PATH = "preprocessed_data"

SSUSA_CSV = "ssusa_cleaned.csv"
IUCN_DATA = "iucn_cleaned.shp"

BUFFER_KM = 1
BUFFER_M = BUFFER_KM * 1000

WGS84 = "EPSG:4326"
AEA = "EPSG:5070"

LON_COL = "Longitude"
LAT_COL = "Latitude"
ARRAY_COL = "Camera_Trap_Array"
SPECIES_COL = "sci_name"

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [93]:
# Load data
# ----------------------------------------------------------
ssusa_file = os.path.join(CLEANED_PATH, SSUSA_CSV)
iucn_file = os.path.join(CLEANED_PATH, IUCN_DATA)

ssusa_df = pd.read_csv(ssusa_file, low_memory=False)
iucn = gpd.read_file(iucn_file)

print("SSUSA rows:", len(ssusa_df))
print("IUCN rows:", len(iucn))

SSUSA rows: 713319
IUCN rows: 733


### Clean and validate IUCN geometries

Before spatial joins, IUCN geometries are checked and repaired.

This section:

- removes null and empty geometries
- repairs invalid geometries
- reprojects the layer to a projected CRS for spatial analysis
- applies a second repair pass after reprojection
- keeps only valid geometries for downstream analysis

In [94]:
# Clean IUCN geometries
# ----------------------------------------------------------
iucn = iucn[
    iucn.geometry.notnull() &
    ~iucn.geometry.is_empty
].copy()

iucn["geometry"] = iucn.geometry.make_valid()

if iucn.crs is None:
    raise ValueError("IUCN shapefile has no CRS defined.")

iucn_aea = iucn.to_crs(AEA)

iucn_aea["geometry"] = iucn_aea.geometry.make_valid()

invalid_mask = ~iucn_aea.geometry.is_valid
if invalid_mask.sum() > 0:
    iucn_aea.loc[invalid_mask, "geometry"] = (
        iucn_aea.loc[invalid_mask, "geometry"].buffer(0)
    )

iucn_aea = iucn_aea[
    iucn_aea.geometry.notnull() &
    ~iucn_aea.geometry.is_empty &
    iucn_aea.geometry.is_valid
].copy()

print("IUCN rows after cleanup:", len(iucn_aea))

/Users/neelima/miniconda3/lib/python3.12/site-packages/shapely/constructive.py:752: RuntimeWarning: invalid value encountered in make_valid
  return lib.make_valid(geometry, **kwargs)


IUCN rows after cleanup: 733


In [ ]:
# print number of unique species and check for ab_thres column
print("Unique species in IUCN:", iucn_aea[SPECIES_COL].nunique())
print("IUCN columns:", iucn_aea.columns.tolist())

# print count of species with abs_thres == True 
print("Before spatial join Species with ab_thres == True:", iucn_aea[iucn_aea["ab_thres"] == True].shape[0])


Unique species in IUCN: 575
IUCN columns: ['id_no', 'sci_name', 'presence', 'origin', 'seasonal', 'category', 'compiler', 'yrcompiled', 'citation', 'subspecies', 'subpop', 'source', 'island', 'tax_comm', 'dist_comm', 'generalisd', 'legend', 'class', 'order', 'family', 'genus', 'marine', 'terrestria', 'freshwater', 'body_mass', 'ab_thres', 'scope_flag', 'geometry']
Species with ab_thres == True: 202


###  Create camera-level 1 km footprints

This section converts each unique camera location into a point geometry,
reprojects to a projected CRS, and creates a 1 km buffer around each point.

Each buffer is saved as a camera footprint and assigned a stable `camera_fp_id`
based on longitude and latitude.


In [95]:
camera_df = (
    ssusa_df[[LON_COL, LAT_COL]]
    .dropna(subset=[LON_COL, LAT_COL])
    .drop_duplicates()
    .copy()
)

print("Unique camera locations:", len(camera_df))

Unique camera locations: 7340


In [96]:
# Create 1 km camera footprints
# ----------------------------------------------------------
camera_points = gpd.GeoDataFrame(
    camera_df,
    geometry=gpd.points_from_xy(camera_df[LON_COL], camera_df[LAT_COL]),
    crs=WGS84
)

camera_points_aea = camera_points.to_crs(AEA)

camera_footprints = camera_points_aea.copy()
camera_footprints["geometry"] = camera_footprints.geometry.buffer(BUFFER_M)

camera_footprints = camera_footprints[
    camera_footprints.geometry.notnull() &
    ~camera_footprints.geometry.is_empty
].copy()

camera_footprints["geometry"] = camera_footprints.geometry.make_valid()

invalid_mask = ~camera_footprints.geometry.is_valid
if invalid_mask.sum() > 0:
    camera_footprints.loc[invalid_mask, "geometry"] = (
        camera_footprints.loc[invalid_mask, "geometry"].buffer(0)
    )

camera_footprints = camera_footprints[
    camera_footprints.geometry.notnull() &
    ~camera_footprints.geometry.is_empty &
    camera_footprints.geometry.is_valid
].copy()

# Create camera footprint ID from coordinates
def make_camera_id(lon, lat):
    return f"{lon:.8f}_{lat:.8f}"

camera_footprints["camera_fp_id"] = camera_footprints.apply(
    lambda row: make_camera_id(row[LON_COL], row[LAT_COL]),
    axis=1
)

print("Camera footprints created:", len(camera_footprints))
print("Duplicate camera_fp_id:", camera_footprints["camera_fp_id"].duplicated().sum())

Camera footprints created: 7340
Duplicate camera_fp_id: 0


In [97]:
# Save camera footprints
camera_fp_path = os.path.join(
    OUTPUT_PATH,
    f"ssusa_camera_footprints_{BUFFER_KM}km.geojson"
)

camera_footprints.to_file(camera_fp_path, driver="GeoJSON")
print("Saved:", camera_fp_path)

Saved: preprocessed_data/ssusa_camera_footprints_1km.geojson


###  Spatial join: IUCN with camera-level footprints

This section identifies which IUCN species ranges intersect each camera footprint.

To avoid saving a very large spatial file with repeated IUCN geometries,
the join result is reduced to a lightweight lookup table containing:

- `camera_fp_id`
- `Longitude`
- `Latitude`
- species name

In [109]:
# Camera-level spatial join
# ----------------------------------------------------------
iucn_small = iucn_aea[[SPECIES_COL,"ab_thres", "geometry"]].copy()

camera_fp_small = camera_footprints[
    ["camera_fp_id", LON_COL, LAT_COL, "geometry"]
].copy()

camera_iucn_join = gpd.sjoin(
    iucn_small,
    camera_fp_small,
    how="inner",
    predicate="intersects"
)

print("Camera-level IUCN intersections:", len(camera_iucn_join))

Camera-level IUCN intersections: 321728


In [111]:
# Spatial join: which IUCN polygons intersect each camera footprint?
camera_iucn_join = gpd.sjoin(
    iucn_small,
    camera_fp_small,
    how="inner",
    predicate="intersects"
)

print("Camera-level IUCN intersections:", len(camera_iucn_join))
# print count of unique species with abs_thres == True after spatial joinp
print("After spatial join Species with ab_thres == True:", camera_iucn_join[camera_iucn_join["ab_thres"] == True][SPECIES_COL].nunique())
camera_iucn_join.head()

Camera-level IUCN intersections: 321728
After spatial join Species with ab_thres == True: 68


,sci_name,ab_thres,geometry,index_right,camera_fp_id,Longitude,Latitude
3,ammospermophilus nelsoni,False,"POLYGON ((-2155993.324 1691652.069, -2155075.5...",99634,-119.99134000_35.36459000,-119.991340,35.364590
3,ammospermophilus nelsoni,False,"POLYGON ((-2155993.324 1691652.069, -2155075.5...",99484,-119.97630000_35.36948000,-119.976300,35.369480
3,ammospermophilus nelsoni,False,"POLYGON ((-2155993.324 1691652.069, -2155075.5...",387614,-119.99464700_35.37250700,-119.994647,35.372507
3,ammospermophilus nelsoni,False,"POLYGON ((-2155993.324 1691652.069, -2155075.5...",615987,-119.99786700_35.37717800,-119.997867,35.377178
3,ammospermophilus nelsoni,False,"POLYGON ((-2155993.324 1691652.069, -2155075.5...",98927,-120.05406000_35.33724000,-120.054060,35.337240


In [100]:
camera_iucn_table = pd.DataFrame(
    camera_iucn_join[
        ["camera_fp_id", LON_COL, LAT_COL, SPECIES_COL]
    ]
).drop_duplicates()

camera_lookup_path = os.path.join(
    OUTPUT_PATH,
    f"iucn_camera_species_lookup_{BUFFER_KM}km.csv"
)

camera_iucn_table.to_csv(camera_lookup_path, index=False)
print("Saved:", camera_lookup_path)

Saved: preprocessed_data/iucn_camera_species_lookup_1km.csv


In [101]:
# Save lightweight camera-species lookup table
camera_lookup_path = os.path.join(
    OUTPUT_PATH,
    f"iucn_camera_species_lookup_{BUFFER_KM}km.csv"
)

camera_iucn_table.to_csv(camera_lookup_path, index=False)

print("Saved:", camera_lookup_path)

Saved: preprocessed_data/iucn_camera_species_lookup_1km.csv


## Create Array Level Footprints

All 1 km camera buffers within the same array are then dissolved into one
union footprint representing the full spatial extent of that array.

In [102]:
array_lookup = (
    ssusa_df[[ARRAY_COL, LON_COL, LAT_COL]]
    .dropna(subset=[LON_COL, LAT_COL])
    .drop_duplicates()
    .copy()
)

camera_buffers_with_arrays = array_lookup.merge(
    camera_footprints[[LON_COL, LAT_COL, "camera_fp_id", "geometry"]],
    on=[LON_COL, LAT_COL],
    how="left"
)

camera_buffers_with_arrays = gpd.GeoDataFrame(
    camera_buffers_with_arrays,
    geometry="geometry",
    crs=camera_footprints.crs
)

camera_buffers_with_arrays = camera_buffers_with_arrays[
    camera_buffers_with_arrays.geometry.notnull() &
    ~camera_buffers_with_arrays.geometry.is_empty
].copy()

array_footprints = (
    camera_buffers_with_arrays[[ARRAY_COL, "geometry"]]
    .dissolve(by=ARRAY_COL)
    .reset_index()
)

array_footprints["geometry"] = array_footprints.geometry.make_valid()

invalid_mask = ~array_footprints.geometry.is_valid
if invalid_mask.sum() > 0:
    array_footprints.loc[invalid_mask, "geometry"] = (
        array_footprints.loc[invalid_mask, "geometry"].buffer(0)
    )

array_footprints = array_footprints[
    array_footprints.geometry.notnull() &
    ~array_footprints.geometry.is_empty &
    array_footprints.geometry.is_valid
].copy()

print("Array footprints created:", len(array_footprints))

array_fp_path = os.path.join(
    OUTPUT_PATH,
    f"ssusa_array_footprints_{BUFFER_KM}km.geojson"
)

array_footprints.to_file(array_fp_path, driver="GeoJSON")
print("Saved:", array_fp_path)

/Users/neelima/miniconda3/lib/python3.12/site-packages/shapely/set_operations.py:553: RuntimeWarning: invalid value encountered in unary_union
  return lib.unary_union(collections, **kwargs)


Array footprints created: 262
Saved: preprocessed_data/ssusa_array_footprints_1km.geojson


### Spatial join: IUCN with array-level footprints

This section intersects dissolved array footprints with IUCN range polygons
to identify which species potentially occur within each array footprint.

As with the camera-level result, the output is saved as a lightweight lookup table
rather than a large geospatial join file.

In [103]:
array_iucn_join = gpd.sjoin(
    iucn_small,
    array_footprints[[ARRAY_COL, "geometry"]],
    how="inner",
    predicate="intersects"
)

print("Array-level IUCN intersections:", len(array_iucn_join))

array_iucn_table = pd.DataFrame(
    array_iucn_join[[ARRAY_COL, SPECIES_COL]]
).drop_duplicates()

array_lookup_path = os.path.join(
    OUTPUT_PATH,
    f"iucn_array_species_lookup_{BUFFER_KM}km.csv"
)

array_iucn_table.to_csv(array_lookup_path, index=False)
print("Saved:", array_lookup_path)

Array-level IUCN intersections: 11928
Saved: preprocessed_data/iucn_array_species_lookup_1km.csv


## Final outputs

At the end of the workflow, the notebook saves:

- camera footprints (`GeoJSON`)
- camera-level species lookup table (`CSV`)
- array footprints (`GeoJSON`)
- array-level species lookup table (`CSV`)

These outputs provide a lightweight and reusable representation of
SSUSA–IUCN spatial overlap.